# Cuaderno 3: ruido real y tres formas de medir la recuperación (VitalDB)

**Autor:** Aly Gallo · Continúa los cuadernos 1 (viabilidad) y 2 (simulación nula).

**Dos pasos:**
- **Paso A. Ruido real.** Con los pacientes que tienen 2 o más extracciones previas en los 7 días antes de operar, mide cuánto varía el BSI entre extracciones. Da un valor realista (una cota superior, porque incluye cambios reales) para el ruido de medición que en el cuaderno 2 era una suposición (0,3 y 0,6).
- **Paso B. Tres métodos para medir la recuperación**, evaluados con el mismo simulador:
  - **M0 (el actual):** fracción de la caída recuperada, con la misma línea base en el predictor y en el cociente.
  - **M1:** nivel final en lugar de cociente (no hay línea base en el resultado).
  - **M2:** cociente, pero el predictor usa una extracción previa y el cociente usa **otra independiente**.

**No usa ningún desenlace real de salud.** La recuperación es simulada. Todo se fija de antemano: semilla 2026; BSI = promedio de z(−log PCR), z(albúmina), z(−log leucocitos).

**Cómo se evalúa cada método:** (1) sesgo en el mundo *sin techo* (Δβ debería ser 0); (2) poder cuando *sí* hay techo, usando como umbral el percentil 95 del mundo sin techo; (3) control con 20 000 pacientes simulados para separar un problema de método de un problema de tamaño de muestra.

In [ ]:
import sys, datetime, json, time, warnings
import numpy as np, pandas as pd
from statsmodels.regression.quantile_regression import QuantReg
warnings.simplefilter("ignore")
print("Fecha:", datetime.date.today(), "| Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

W = 7.0; PRE_DIAS = 7; MIN_POST = 3
SEMILLA = 2026
N_REPS = 150                  # repeticiones por escenario
TECHOS = [0.5, 1.0]           # techo moderado y fuerte
ACOPLAMIENTOS = [0.0, 0.3]
D0, T0, TAU = 1.0, 2.0, 2.0
UMBRAL_CAIDA = 0.5
pruebas = ["crp", "alb", "wbc"]


## Paso 1. Datos y cohorte real (igual que en el cuaderno 2)

In [ ]:
lab = pd.read_csv("https://api.vitaldb.net/labs")
cases = pd.read_csv("https://api.vitaldb.net/cases")
d = lab.merge(cases[["caseid", "subjectid", "opend"]], on="caseid", how="left")
d["dias_post"] = (d.dt - d.opend) / 86400
d["dias_pre"] = -d.dt / 86400

conjuntos = []
for t in pruebas:
    x = d[d.name == t]
    pre = set(x[(x.dt <= 0) & (x.dias_pre <= PRE_DIAS)].caseid)
    p = x[(x.dias_post >= 0) & (x.dias_post <= W)].groupby("caseid").dias_post.agg(["size", "max"])
    post = set(p[(p["size"] >= MIN_POST) & (p["max"] >= W / 2)].index)
    conjuntos.append(pre & post)
coh = set.intersection(*conjuntos)
c = cases[cases.caseid.isin(coh)].sort_values("caseid").drop_duplicates("subjectid", keep="first")
ids = c.caseid.tolist()
print("Cirugias en la cohorte:", len(coh), "| pacientes unicos:", len(ids), "(esperado: 774 y 738)")

base = {}
for t in pruebas:
    x = d[(d.name == t) & d.caseid.isin(ids) & (d.dt <= 0) & (d.dias_pre <= PRE_DIAS)]
    base[t] = x.sort_values("dt").groupby("caseid").result.last()
B = pd.DataFrame(base).loc[ids]
# parametros de estandarizacion (se reutilizan en el Paso A)
def comp(df):
    return pd.concat([-np.log(df.crp + 0.1), df.alb, -np.log(df.wbc)], axis=1)
Z = comp(B)
mu_z, sd_z = Z.mean(), Z.std()
raw = ((Z - mu_z) / sd_z).mean(axis=1)
mu_raw, sd_raw = raw.mean(), raw.std()
bsi_real = ((raw - mu_raw) / sd_raw).values

xc = d[(d.name == "crp") & d.caseid.isin(ids) & (d.dias_post >= 0) & (d.dias_post <= W)]
tiempos = [np.sort(xc[xc.caseid == i].dias_post.values) for i in ids]
m = max(len(t) for t in tiempos)
T = np.full((len(ids), m), np.nan)
for k, t in enumerate(tiempos):
    T[k, :len(t)] = t
N = len(ids)


## Paso A. Ruido real del BSI antes de la cirugía
VitalDB casi nunca trae las tres pruebas en el mismo instante, así que el ruido se estima **prueba por prueba**: en cada prueba se comparan las dos últimas extracciones previas (≤ 7 días antes de operar) de la misma cirugía. La desviación de las diferencias dividida por √2 da el ruido de una medición. Luego se combina para el BSI (promedio de 3 componentes independientes). Es una **cota superior**: incluye cambios clínicos reales entre extracciones.

In [ ]:
filas_a = []; comp_sd = {}
for t in pruebas:
    x = d[(d.name == t) & (d.dt <= 0) & (d.dias_pre <= PRE_DIAS)].sort_values(["caseid", "dt"])
    n_ext = x.groupby("caseid").dt.nunique()
    dos = n_ext[n_ext >= 2].index
    ult = x[x.caseid.isin(dos)].groupby("caseid").tail(2).copy()
    v = ult.result
    if t == "crp": v = -np.log(v + 0.1)
    if t == "wbc": v = -np.log(v)
    ult["v"] = v
    g = ult.groupby("caseid")
    dif = g.v.apply(lambda s: s.iloc[1] - s.iloc[0])
    gap = g.dt.apply(lambda s: (s.iloc[1] - s.iloc[0]) / 86400)
    sd_t = dif.std() / np.sqrt(2) if len(dif) > 5 else np.nan
    sd_t2 = dif[gap <= 2].std() / np.sqrt(2) if (gap <= 2).sum() > 5 else np.nan
    comp_sd[t] = (sd_t, sd_t2)
    filas_a.append((t, int((n_ext >= 1).sum()), int((n_ext >= 2).sum()), int(len(set(dos) & set(ids))), round(float(sd_t), 3) if not np.isnan(sd_t) else None))
print(pd.DataFrame(filas_a, columns=["prueba", "cirugias con >=1", "cirugias con >=2", "de ellas en cohorte", "SD ruido (unidad transformada)"]).to_string(index=False))

# cohorte con 2 extracciones previas en las TRES pruebas (lo que necesitaria M2 en la practica)
dos_todas = None
for t in pruebas:
    x = d[(d.name == t) & (d.dt <= 0) & (d.dias_pre <= PRE_DIAS)]
    s_ = set(x.groupby("caseid").dt.nunique().loc[lambda s: s >= 2].index)
    dos_todas = s_ if dos_todas is None else dos_todas & s_
print("\nCirugias con >=2 extracciones previas en las 3 pruebas:", len(dos_todas), "| en la cohorte:", len(dos_todas & set(ids)))

sdz = Z.std().values
def sigma_bsi(k):
    sdc = np.array([comp_sd[t][k] for t in pruebas]) / sdz
    return float(np.sqrt((sdc ** 2).sum()) / 3 / sd_raw) if not np.isnan(sdc).any() else np.nan
s_all, s_2d = sigma_bsi(0), sigma_bsi(1)
print("Ruido del BSI (todas las parejas):", None if np.isnan(s_all) else round(s_all, 2))
print("Ruido del BSI (parejas separadas <= 2 dias):", None if np.isnan(s_2d) else round(s_2d, 2))
cand = [s for s in [s_all, s_2d] if not np.isnan(s)]
if cand:
    SIGMA_HAT = float(np.round(np.median(cand), 2)); origen = "medido"
else:
    SIGMA_HAT = 0.3; origen = "NO se pudo medir; se usa 0.3 por defecto"
print("\nRUIDO USADO EN EL PASO B:", SIGMA_HAT, "(" + origen + ")")
RUIDOS = sorted(set([0.15, 0.3, round(min(SIGMA_HAT, 1.0), 2)]))
print("Rejilla de ruidos para el Paso B:", RUIDOS)


## Paso B. Simulador con los tres métodos
Cada paciente simulado tiene dos mediciones previas independientes (`base1` y `base2`) del mismo estado verdadero:
- **M0:** predictor `base1`; resultado `(final − nadir)/(base1 − nadir)`.
- **M1:** predictor `base1`; resultado `final` (nivel final, sin cociente).
- **M2:** predictor `base1`; resultado `(final − nadir)/(base2 − nadir)`.
En todos, la prueba es Δβ = β(0,90) − β(0,50) de una regresión cuantílica de resultado sobre el predictor. Se analizan solo pacientes con caída observada > 0,5 SD (mismo criterio para M0 y M1 con `base1`; M2 con `base2`).

In [ ]:
def generar(rng, idx, ruido, acople, techo):
    n = len(idx); Tn = T[idx]; mask = ~np.isnan(Tn)
    Bt = bsi_real[idx]
    D = np.clip(D0 * (1 + acople * Bt), 0.3, None)
    u = rng.beta(4, 2, n)
    r = u if techo is None else np.minimum(u, np.clip(0.95 + techo * Bt, 0.05, 1.0))
    k = 1 - np.exp(-(W - T0) / TAU)
    t = np.where(mask, Tn, 0.0)
    rec = r[:, None] * (1 - np.exp(-(t - T0) / TAU)) / k
    q = np.where(t <= T0, t / T0, 1 - rec)
    x = Bt[:, None] - D[:, None] * q
    obs = np.where(mask, x + rng.normal(0, ruido, x.shape), np.nan)
    base1 = Bt + rng.normal(0, ruido, n)
    base2 = Bt + rng.normal(0, ruido, n)
    srt = np.sort(np.where(mask, obs, np.inf), axis=1)
    nadir = srt[:, :2].mean(axis=1)
    last = np.nanargmax(np.where(mask, Tn, -np.inf), axis=1)
    fin = obs[np.arange(n), last]
    return base1, base2, nadir, fin

def resultado(metodo, base1, base2, nadir, fin):
    if metodo == "M0":
        ok = (base1 - nadir) > UMBRAL_CAIDA
        R = np.clip((fin - nadir) / np.where(ok, base1 - nadir, 1), -0.5, 1.5)
    elif metodo == "M1":
        ok = (base1 - nadir) > UMBRAL_CAIDA
        R = fin
    else:  # M2
        ok = (base2 - nadir) > UMBRAL_CAIDA
        R = np.clip((fin - nadir) / np.where(ok, base2 - nadir, 1), -0.5, 1.5)
    return base1[ok], R[ok]

def delta_beta(S, R):
    X = np.column_stack([np.ones(len(S)), S])
    b90 = QuantReg(R, X).fit(q=0.90, max_iter=1000).params[1]
    b50 = QuantReg(R, X).fit(q=0.50, max_iter=1000).params[1]
    return b90 - b50


## Paso B1. Comparación de métodos con la cohorte real (n = 738)
Tarda unos minutos.

In [ ]:
rng = np.random.default_rng(SEMILLA)
METODOS = ["M0", "M1", "M2"]
filas = []
t_ini = time.time()
for ruido in RUIDOS:
    for acople in ACOPLAMIENTOS:
        for met in METODOS:
            nulo = []
            for _ in range(N_REPS):
                idx = rng.choice(N, N, replace=True)           # remuestreo de pacientes
                S, R = resultado(met, *generar(rng, idx, ruido, acople, None))
                nulo.append(delta_beta(S, R))
            nulo = np.array(nulo); crit = np.percentile(nulo, 95)
            fila = {"ruido": ruido, "acople": acople, "metodo": met,
                    "Δβ nulo, media": round(nulo.mean(), 3), "critico (p95)": round(crit, 3)}
            for tk in TECHOS:
                alt = []
                for _ in range(N_REPS):
                    idx = rng.choice(N, N, replace=True)
                    S, R = resultado(met, *generar(rng, idx, ruido, acople, tk))
                    alt.append(delta_beta(S, R))
                fila[f"poder techo {tk}"] = round(float(np.mean(np.array(alt) > crit)), 2)
            filas.append(fila)
    print("ruido", ruido, "listo;", round(time.time() - t_ini), "s")
comparacion = pd.DataFrame(filas)
print(comparacion.to_string(index=False))


## Paso B2. Control con 20 000 pacientes simulados
Si un método da Δβ ≠ 0 sin techo, o no distingue el techo ni con 20 000, el problema es del método.

In [ ]:
rng2 = np.random.default_rng(SEMILLA + 1)
M_GRANDE = 20000
sel = rng2.integers(0, N, M_GRANDE)
T_real, bsi_orig = T.copy(), bsi_real.copy()
T, bsi_real = T_real[sel], bsi_orig[sel]
filas_c = []
for ruido in RUIDOS:
    for acople in ACOPLAMIENTOS:
        for met in METODOS:
            fila = {"ruido": ruido, "acople": acople, "metodo": met}
            for nombre, tk in [("sin techo", None), ("techo 0.5", 0.5), ("techo 1.0", 1.0)]:
                S, R = resultado(met, *generar(rng2, np.arange(M_GRANDE), ruido, acople, tk))
                fila["Δβ " + nombre] = round(delta_beta(S, R), 3)
            filas_c.append(fila)
T, bsi_real = T_real, bsi_orig
control = pd.DataFrame(filas_c)
print(control.to_string(index=False))


## Paso 5. Guardar

In [ ]:
comparacion.to_csv("sim3_comparacion_metodos.csv", index=False)
control.to_csv("sim3_control_n20000.csv", index=False)
with open("sim3_parametros.json", "w") as f:
    json.dump({"fecha": str(datetime.date.today()), "semilla": SEMILLA, "n_reps": N_REPS, "cohorte_n": int(N),
               "sigma_hat": SIGMA_HAT, "sigma_origen": origen, "ruidos": RUIDOS, "acoplamientos": ACOPLAMIENTOS, "techos": TECHOS,
               "W": W, "D0": D0, "T0": T0, "TAU": TAU, "umbral_caida": UMBRAL_CAIDA}, f, indent=2)
print("Guardado: sim3_comparacion_metodos.csv, sim3_control_n20000.csv y sim3_parametros.json")


## Cómo leer los resultados
- **Paso A:** el ruido estimado es una cota superior. Si sale cercano a 0,3 o más, los escenarios ruidosos del cuaderno 2 eran realistas; si sale mucho menor, eran pesimistas.
- **Δβ nulo, media (sin techo):** lo sano es ≈ 0. Un valor negativo o positivo sostenido indica que el método fabrica un techo falso o lo esconde.
- **Poder:** se busca ≥ 0,80 para techo moderado o fuerte.
- **Control con 20 000:** un método sano muestra Δβ ≈ 0 sin techo y claramente positivo con techo, en todos los ruidos y acoplamientos razonables.
- **M2 en la práctica** exige que cada paciente tenga 2 extracciones previas completas. El Paso A dice cuántos cumplen (el conteo dentro de la cohorte).

**Límites:** la forma del techo y de la recuperación son suposiciones; el ruido real incluye cambios clínicos; es una simulación, no un resultado clínico.